In [45]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv
from torch_geometric.loader import LinkNeighborLoader
from torch_geometric.nn import GraphSAGE
from sklearn.linear_model import LogisticRegression
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [46]:
# Load the datasets
cora_dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = cora_dataset[0]
data = data.to(device, 'x', 'edge_index')

train_loader = LinkNeighborLoader(
    data,
    batch_size=256,
    shuffle=True,
    neg_sampling_ratio=1.0,
    num_neighbors=[10, 10],
)

model = GraphSAGE(
    data.num_node_features,
    hidden_channels=64,
    num_layers=2,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

/home/wcorcoran/anaconda3/envs/mesp/lib/python3.8/site-packages/torch_geometric/sampler/neighbor_sampler.py:61: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  warnings.warn(f"Using '{self.__class__.__name__}' without a "


In [47]:
def train():
    model.train()

    total_loss = 0
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        h = model(batch.x, batch.edge_index)
        h_src = h[batch.edge_label_index[0]]
        h_dst = h[batch.edge_label_index[1]]
        pred = (h_src * h_dst).sum(dim=-1)
        loss = F.binary_cross_entropy_with_logits(pred, batch.edge_label)
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * pred.size(0)

    return total_loss / data.num_nodes

In [48]:
@torch.no_grad()
def test():
    model.eval()
    out = model(data.x, data.edge_index).cpu()

    clf = LogisticRegression()
    clf.fit(out[data.train_mask], data.y[data.train_mask])

    val_acc = clf.score(out[data.val_mask], data.y[data.val_mask])
    test_acc = clf.score(out[data.test_mask], data.y[data.test_mask])

    return val_acc, test_acc

In [49]:
for epoch in range(0, 200):
    loss = train()
    acc = test()[1]
    print(f'Epoch: {epoch:03d}, Loss: {loss:.4f}, Accuracy: {acc:.4f}')

Epoch: 000, Loss: 4.8859, Accuracy: 0.4270
Epoch: 001, Loss: 4.2669, Accuracy: 0.4620
Epoch: 002, Loss: 4.0499, Accuracy: 0.4950
Epoch: 003, Loss: 3.9819, Accuracy: 0.4880
Epoch: 004, Loss: 3.9001, Accuracy: 0.5670
Epoch: 005, Loss: 3.8343, Accuracy: 0.5260
Epoch: 006, Loss: 3.7695, Accuracy: 0.5970
Epoch: 007, Loss: 3.7793, Accuracy: 0.5580
Epoch: 008, Loss: 3.7786, Accuracy: 0.5780
Epoch: 009, Loss: 3.7988, Accuracy: 0.5820
Epoch: 010, Loss: 3.7108, Accuracy: 0.5840
Epoch: 011, Loss: 3.7177, Accuracy: 0.5570
Epoch: 012, Loss: 3.7373, Accuracy: 0.5540
Epoch: 013, Loss: 3.7067, Accuracy: 0.5510
Epoch: 014, Loss: 3.7463, Accuracy: 0.5450
Epoch: 015, Loss: 3.7385, Accuracy: 0.5480
Epoch: 016, Loss: 3.6638, Accuracy: 0.5640
Epoch: 017, Loss: 3.7189, Accuracy: 0.5780
Epoch: 018, Loss: 3.6813, Accuracy: 0.5560
Epoch: 019, Loss: 3.7291, Accuracy: 0.5530
Epoch: 020, Loss: 3.6752, Accuracy: 0.5420
Epoch: 021, Loss: 3.6565, Accuracy: 0.5290
Epoch: 022, Loss: 3.7381, Accuracy: 0.5390
Epoch: 023,

In [50]:
torch.save(model.state_dict(), 'cora_gsage.pt')